# 3. Baseline: ResNet-18 fine-tuned on grey photographs (Kaggle, GPU T4)

**Settings:** Accelerator **GPU T4 x1**. Internet can stay **off** (the ImageNet weights are in the dataset).
Attach the uploaded dataset *banglahand-grey-cnn* (Add data -> Your datasets), then **Save Version -> Save & Run All**.

What it does: fine-tunes ResNet-18 (layer4 + new head, 4 epochs, AdamW) on 224-px tiles of the grey photographs of the
training writers in each fold, over the same 20 repetitions of writer-independent 5-fold cross-validation as the paper,
and reports writer balanced accuracy end to end, with a linear SVM on the fine-tuned features, and after regressing
ink density and photo intensity out of those features.

Runtime: about 1 h on a T4 (the first few minutes cut the tiles). Download **`/kaggle/working/grey_cnn_results.zip`** at the end and send it back.

In [ ]:
import os, sys, glob, shutil, subprocess, time
from pathlib import Path
KAGGLE = os.environ.get("KAGGLE_ROOT", "/kaggle")          # override only for local testing
hits = glob.glob(f"{KAGGLE}/input/**/dataset/metadata/images.csv", recursive=True)
assert hits, "Attach the uploaded dataset to this notebook (Add data -> Your datasets)."
INPUT = Path(hits[0]).parents[2]
PROJECT = Path(f"{KAGGLE}/working/project"); PROJECT.mkdir(parents=True, exist_ok=True)
shutil.rmtree(PROJECT / "scripts", ignore_errors=True)        # always use the code of the attached dataset version
shutil.copytree(INPUT / "code", PROJECT / "scripts")
assert "finetune_grey_embeddings" in (PROJECT / "scripts/run_finetuned_grey_cnn.py").read_text(), \
    "Old dataset version attached: upload the new banglahand-grey-cnn.zip as a new dataset version."
for f in ["finetune_grey_metrics_by_split.csv", "finetune_grey_writer_predictions.csv", "finetune_grey_embeddings.npz"]:
    (PROJECT / "results/final42_grey" / f).unlink(missing_ok=True)   # no stale outputs from an earlier run
if not (PROJECT / "dataset").exists():
    os.symlink(INPUT / "dataset", PROJECT / "dataset")
RUN = PROJECT / "results" / "final42_grey"
(RUN / "features").mkdir(parents=True, exist_ok=True)
for f in ["features.npz", "descriptors.csv"]:
    if not (RUN / "features" / f).exists():
        shutil.copy(INPUT / "extra/run_features" / f, RUN / "features" / f)
os.environ["TORCH_HOME"] = str(INPUT / "weights/torch")
import torch; print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("input:", INPUT, "| run folder:", RUN)

In [ ]:
t = time.time()
p = subprocess.Popen([sys.executable, "scripts/run_finetuned_grey_cnn.py", "--run", str(RUN)], cwd=PROJECT,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
with open(RUN / "log_grey_cnn.txt", "w") as log:
    for line in p.stdout:
        log.write(line)
        if "done" in line or "grey tiles" in line or "ba_" in line or "mean" in line or "Error" in line:
            print(line.rstrip(), flush=True)
assert p.wait() == 0, "failed, see " + str(RUN / "log_grey_cnn.txt")
print(f"finished in {(time.time() - t) / 60:.0f} min")

In [ ]:
import pandas as pd
m = pd.read_csv(RUN / "finetune_grey_metrics_by_split.csv")
display((100 * m.drop(columns="seed").agg(["mean", "std"])).round(1))

In [ ]:
keep = ["finetune_grey_writer_predictions.csv", "finetune_grey_metrics_by_split.csv", "finetune_grey_provenance.json",
        "finetune_grey_embeddings.npz", "log_grey_cnn.txt"]
with __import__("zipfile").ZipFile(f"{KAGGLE}/working/grey_cnn_results.zip", "w") as z:
    for f in keep:
        z.write(RUN / f, f)
print("download:", f"{KAGGLE}/working/grey_cnn_results.zip")